# Week 9 Assignment — A Conversational Agent with Memory in LangChain

**Module 9: Memory in Agentic Systems**

**Name:** Nagarajan Kanagamani

## Objective

Build a simple conversational agent in LangChain that **remembers earlier turns** of the conversation and uses them to give context-aware replies. The agent has:

* a **chat model** (`ChatOpenAI`, temperature 0.2),
* a **conversation memory** that stores every prior user/assistant turn, and
* one **tool** (a calculator) that it calls only when arithmetic is actually needed.

This notebook follows the six steps of the assignment and answers Tasks 1–5:

| Task | Where |
|---|---|
| Task 1 — Scenario & key detail | Section *Task 1* |
| Task 2 — Chat model + memory | Steps 2 & 3 |
| Task 3 — Register one tool | Step 4 |
| Task 4 — 3–4 turn recall demo | Step 6 |
| Task 5 — Reflection | Final section |

> **Note on API keys:** All keys are read from a `.env` file (see `Week7/env_example`) via `python-dotenv`. No key is hard-coded in this notebook.

## Task 1 — Scenario: *Travel Buddy*

The agent is a friendly **travel buddy** that helps the user plan their next trip.

**Key details the agent must remember** (mentioned once, early in the chat, and never repeated by the user):

1. **Last destination** — the user recently visited **Kyoto, Japan** and loved its temples and quiet gardens.
2. **Dietary preference** — the user is **vegetarian**.
3. **Trip budget** — **USD 1,500 for 5 days** (the agent uses the calculator to turn this into a per-day budget).

Later in the conversation the user asks for a recommendation *"similar to my last trip"* and finally asks the agent to *remind* them of their details. A memory-less model cannot answer either question correctly — it has never "seen" Kyoto in the current request. With memory, it should recall Kyoto, stay vegetarian-friendly, and respect the budget.

## Step 1 — Environment Setup

Requires **Python 3.9+**. Install the libraries:

```
pip install langchain langchain-openai python-dotenv
```

* `langchain` / `langchain-core` — prompts, messages, tools and chat-history (memory) classes
* `langchain-openai` — the `ChatOpenAI` chat-model wrapper (also works with Together AI's OpenAI-compatible endpoint)
* `python-dotenv` — loads `OPENAI_API_KEY` / `TOGETHER_API_KEY` from a `.env` file

In [ ]:
# Run once. Comment out after the first successful install to save time on reruns.
%pip install -q langchain langchain-openai python-dotenv

In [ ]:
import sys
print(f"Python version: {sys.version}")
assert sys.version_info >= (3, 9), "This assignment requires Python 3.9 or newer."

In [ ]:
import os
from dotenv import load_dotenv

# Loads variables from a local .env file (copy env_example -> .env and fill in your keys).
load_dotenv()

OPENAI_API_KEY   = os.getenv("OPENAI_API_KEY")
TOGETHER_API_KEY = os.getenv("TOGETHER_API_KEY")   # course default provider, OpenAI-compatible endpoint

if not (OPENAI_API_KEY or TOGETHER_API_KEY):
    raise EnvironmentError(
        "No LLM key found. Set OPENAI_API_KEY (for OpenAI) or TOGETHER_API_KEY "
        "(for Together's OpenAI-compatible endpoint) in your .env file."
    )

print("OPENAI_API_KEY  :", "found" if OPENAI_API_KEY else "not set")
print("TOGETHER_API_KEY:", "found" if TOGETHER_API_KEY else "not set")
print("Environment ready")

## Step 2 — Initialise the LLM

We use `ChatOpenAI` with a **low temperature (0.2)**. Low temperature keeps answers consistent from run to run, which matters for memory: we want the model to *faithfully repeat* facts from history rather than creatively paraphrase (or invent) them.

If `OPENAI_API_KEY` is set we use OpenAI directly; otherwise we point `ChatOpenAI` at Together AI's OpenAI-compatible endpoint (the course default provider).

In [ ]:
from langchain_openai import ChatOpenAI

if OPENAI_API_KEY:
    llm = ChatOpenAI(
        model=os.getenv("LLM_MODEL_OPENAI", "gpt-4o-mini"),
        temperature=0.2,
        api_key=OPENAI_API_KEY,
    )
    print("Using OpenAI as the LLM provider.")
else:
    llm = ChatOpenAI(
        model=os.getenv("LLM_MODEL_DEFAULT", "meta-llama/Llama-3.3-70B-Instruct-Turbo"),
        temperature=0.2,
        api_key=TOGETHER_API_KEY,
        base_url="https://api.together.xyz/v1",
    )
    print("Using Together AI as the LLM provider.")

# One round-trip to confirm the key and setup work before adding memory.
print(llm.invoke("Reply with the single word: ready").content)

### Baseline: the raw model has no memory

Each `llm.invoke(...)` call is independent. Below we tell the model a fact, then ask about it in a *separate* call. The model cannot know the answer — this is the problem memory solves.

In [ ]:
print("Call 1:", llm.invoke("Hi! I just got back from a trip to Kyoto.").content[:200], "...\n")
print("Call 2:", llm.invoke("Where did I travel last?").content)

## Step 3 — Add Memory

LangChain's `ConversationBufferMemory` is a thin wrapper around a **chat message history** that keeps *every* turn in a buffer. In recent LangChain releases (0.3 / 1.x) the legacy `ConversationBufferMemory` class has been deprecated in favour of the underlying `BaseChatMessageHistory` objects, so we use `InMemoryChatMessageHistory` from `langchain_core` directly — it is exactly the buffer `ConversationBufferMemory` stores internally, and it works on every LangChain version.

We keep **one history per session id**, so several independent conversations can run side by side without leaking into each other.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory

# session_id -> conversation buffer (equivalent to ConversationBufferMemory's chat_memory)
memory_store: dict[str, InMemoryChatMessageHistory] = {}

def get_memory(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in memory_store:
        memory_store[session_id] = InMemoryChatMessageHistory()
    return memory_store[session_id]

def show_memory(session_id: str) -> None:
    msgs = get_memory(session_id).messages
    print(f"Memory for session '{session_id}' — {len(msgs)} message(s):")
    for m in msgs:
        print(f"  [{m.type:>5}] {m.content[:110]}")

**Verify that memory stores prior turns and is accessible.** We write two turns into a test session, read them back, and then pass the stored history to the model to confirm it can use it.

In [ ]:
from langchain_core.messages import HumanMessage

test_mem = get_memory("memory-test")
test_mem.add_user_message("My favourite city is Kyoto.")
test_mem.add_ai_message("Lovely choice — Kyoto's temples are beautiful!")

show_memory("memory-test")

# The stored turns are plain message objects, so they can be fed straight back to the model.
reply = llm.invoke(test_mem.messages + [HumanMessage("What is my favourite city? Answer in one word.")])
print("\nModel with memory attached ->", reply.content)

## Step 4 — Register One Tool: Calculator

The agent sometimes needs exact arithmetic (e.g. splitting a budget across days). LLMs are unreliable at arithmetic, so we expose a small calculator tool.

**Schema** — one input, one output:

| Field | Type | Description |
|---|---|---|
| `expression` (input) | `str` | An arithmetic expression, e.g. `"1500 / 5"` |
| result (output) | `str` | The numeric result, or an error message |

The expression is evaluated with Python's `ast` module and only `+ - * / // % **` and parentheses are allowed — **no `eval()`**, so the model cannot run arbitrary code through the tool.

In [ ]:
import ast
import operator
from pydantic import BaseModel, Field
from langchain_core.tools import tool

_ALLOWED_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv, ast.Mod: operator.mod,
    ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos,
}

def _safe_eval(node):
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("Only numbers and + - * / // % ** ( ) are allowed.")


class CalculatorInput(BaseModel):
    expression: str = Field(description="An arithmetic expression to evaluate, e.g. '1500 / 5'.")


@tool("calculator", args_schema=CalculatorInput)
def calculator(expression: str) -> str:
    "Evaluate a basic arithmetic expression and return the numeric result. Use for any budget or number maths."
    try:
        result = _safe_eval(ast.parse(expression.replace(",", ""), mode="eval"))
        return str(round(result, 4))
    except Exception as e:
        return f"Error: {e}"

**Test the tool on its own** before wiring it into the agent:

In [ ]:
print("Schema :", calculator.args)
for expr in ["1500 / 5", "(1500 - 450) / 5", "2 ** 10", "__import__('os')"]:
    print(f"{expr!r:>22} -> {calculator.invoke({'expression': expr})}")

## Step 5 — Build the Agent

The agent combines the three pieces:

1. **System prompt** — concise; tells the model to use the conversation history, call the calculator *only* for arithmetic, and ask a clarifying question if unsure.
2. **Memory** — the session's full message history is inserted between the system prompt and the new user message on every turn.
3. **Tool calling** — the model is bound to the calculator with `llm.bind_tools(...)`. If the model emits a tool call we execute it, feed the `ToolMessage` result back, and let the model continue until it produces a final answer.

After each turn, the user message and the final answer are **saved back into memory** (just like `ConversationBufferMemory.save_context`). Intermediate tool messages are kept out of long-term memory to keep the buffer small and readable.

In [ ]:
from langchain_core.messages import SystemMessage, ToolMessage

SYSTEM_PROMPT = (
    "You are 'Wanderly', a friendly travel-buddy assistant.\n"
    "- The conversation history is your memory. Always check it for the user's name, past trips, "
    "preferences (e.g. diet) and budget, and use those details in your answers without asking again.\n"
    "- If the user refers to something earlier ('my last trip', 'that budget'), resolve it from the history.\n"
    "- Use the calculator tool ONLY when exact arithmetic is needed; never for general questions.\n"
    "- If a needed detail is not in the history, ask a short clarifying question instead of guessing.\n"
    "- Keep replies concise: at most 5 short sentences or bullet points."
)

tools = [calculator]
tools_by_name = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)


def chat(user_input: str, session_id: str = "travel-demo", use_memory: bool = True, verbose: bool = True) -> str:
    memory = get_memory(session_id)
    history = memory.messages if use_memory else []
    messages = [SystemMessage(SYSTEM_PROMPT), *history, HumanMessage(user_input)]

    # Agent loop: let the model call tools until it gives a final answer (max 5 steps).
    for _ in range(5):
        ai_msg = llm_with_tools.invoke(messages)
        messages.append(ai_msg)
        if not ai_msg.tool_calls:
            break
        for tc in ai_msg.tool_calls:
            result = tools_by_name[tc["name"]].invoke(tc["args"])
            if verbose:
                print(f"   [tool] {tc['name']}({tc['args']}) -> {result}")
            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    answer = ai_msg.content
    if use_memory:
        memory.add_user_message(user_input)
        memory.add_ai_message(answer)
    return answer


def turn(n: int, user_input: str, **kwargs) -> str:
    print(f"--- Turn {n} ---------------------------------------------")
    print(f"User : {user_input}")
    answer = chat(user_input, **kwargs)
    print(f"Agent: {answer}\n")
    return answer

**Verify the tool is callable from inside the agent.** We use a throw-away session so the main demo starts with empty memory.

In [ ]:
_ = turn(0, "Quick check: what is 1234 * 56?", session_id="tool-check")

## Step 6 — Run and Observe (Task 4: 4-turn demo)

The user shares their details in **Turn 1** and **Turn 2**. **Turn 3** and **Turn 4** only make sense if the agent remembers them:

* Turn 3 says *"similar to my last trip"* and *"my daily budget"* without restating either.
* Turn 4 asks the agent to recall everything.

In [ ]:
SESSION = "travel-demo"
memory_store.pop(SESSION, None)   # start fresh so the demo is reproducible

t1 = turn(1, "Hi, I'm Nagarajan! I just got back from Kyoto, Japan — I loved the temples and quiet gardens. "
             "By the way, I'm vegetarian.", session_id=SESSION)

t2 = turn(2, "For my next trip I have a budget of 1500 USD for 5 days. How much is that per day?",
          session_id=SESSION)

t3 = turn(3, "Suggest one destination similar to my last trip that fits my daily budget, "
             "and one local dish I could eat there.", session_id=SESSION)

t4 = turn(4, "Before I go — remind me: what's my name, where did I travel last, "
             "what's my diet, and what's my daily budget?", session_id=SESSION)

### What is in memory now?

The buffer holds all 8 messages (4 user + 4 assistant). This is exactly what the model sees as context on the next turn.

In [ ]:
show_memory(SESSION)

### Automatic recall check

A simple keyword check on the final turn confirms that each remembered detail actually appears in the answer. Any detail that is missing is recorded as a **recall failure**.

In [ ]:
expected = {
    "name (Nagarajan)": ["nagarajan"],
    "last trip (Kyoto)": ["kyoto"],
    "diet (vegetarian)": ["vegetarian"],
    "daily budget (300)": ["300"],
}
answer_lc = t4.lower()
failures = []
for detail, keywords in expected.items():
    ok = any(k in answer_lc for k in keywords)
    print(f"{'PASS' if ok else 'FAIL'}  {detail}")
    if not ok:
        failures.append(detail)

print("\nRecall failures:", failures or "none")

### Comparison: the same Turn 4 *without* memory

Here we send the identical Turn 4 question but with `use_memory=False` (the history is not passed in). The agent should either admit it doesn't know or ask a clarifying question — as instructed by the system prompt — rather than recall the details.

In [ ]:
_ = turn(4, "Before I go — remind me: what's my name, where did I travel last, "
            "what's my diet, and what's my daily budget?", session_id=SESSION, use_memory=False)

### Observing a recall failure — and fixing it

A common memory configuration is a **window** that keeps only the last *k* messages to save tokens. Below we simulate a window of the last **2 messages** (just Turn 4 itself). Kyoto, the vegetarian diet and the name were mentioned in Turn 1, so they have fallen out of the window and the agent **fails to recall them**.

The fix is to keep the full buffer (as in the main demo) or, for long conversations, to keep a **summary / key-facts** message pinned at the top of the history. We demonstrate the key-facts fix here.

In [ ]:
from typing import Optional
from langchain_core.messages import AIMessage

def windowed_session(source: str, target: str, k: int, pinned_facts: Optional[str] = None) -> None:
    "Copy the last k messages of `source` into a fresh `target` session, optionally pinning a facts summary."
    memory_store.pop(target, None)
    mem = get_memory(target)
    if pinned_facts:
        mem.add_message(AIMessage(f"[Saved user facts] {pinned_facts}"))
    mem.add_messages(get_memory(source).messages[-k:])

question = "Where did I travel last, and is there any food restriction I mentioned?"

print(">>> Window of 2 messages, no pinned facts (expected to FAIL):")
windowed_session(SESSION, "window-k2", k=2)
_ = turn(5, question, session_id="window-k2")

print(">>> Window of 2 messages + pinned key facts (expected to PASS):")
windowed_session(SESSION, "window-k2-facts", k=2,
                 pinned_facts="Name: Nagarajan; last trip: Kyoto, Japan (loved temples & gardens); "
                              "diet: vegetarian; budget: 1500 USD for 5 days (300 USD/day).")
_ = turn(5, question, session_id="window-k2-facts")

### Observation log

| # | Observation | Adjustment |
|---|---|---|
| 1 | The raw `llm.invoke` baseline could not answer *"Where did I travel last?"* — every call is stateless. | Attached a per-session chat-message buffer and passed it into every call. |
| 2 | Risk: with a vague system prompt the model may re-ask for the budget in Turn 3 instead of looking it up. | The system prompt explicitly says *check the history for past trips, diet and budget* and *resolve references like "my last trip" from the history*. |
| 3 | Risk: at higher temperature recalled numbers can be paraphrased or re-computed differently. | `temperature=0.2`, and the calculator produces the exact per-day figure, which is then stored in memory. |
| 4 | A 2-message window dropped Kyoto and the vegetarian preference (see the cell above). | Use the full buffer for short chats; for long chats pin a key-facts summary at the top of memory. |

*(Re-run the demo cells and update this table if your run behaves differently — LLM output can vary slightly between runs and providers.)*

## Task 5 — Reflection: How Memory Changed the Agent's Usefulness

**Without memory**, the agent is a one-shot Q&A box. Each message is answered in isolation, so the user must restate their name, last destination, diet and budget every time. Requests like *"somewhere similar to my last trip"* are simply unanswerable — the model either guesses or asks again.

**With memory**, the same model behaves like a real travel buddy:

* It **resolved references** ("my last trip" → Kyoto; "my daily budget" → 300 USD) without the user repeating them.
* It **combined facts from different turns** — the Turn 3 suggestion used the Kyoto preference (Turn 1), the vegetarian diet (Turn 1) and the calculated budget (Turn 2) together.
* The **tool and memory worked together**: the calculator's exact result was stored in the conversation, so later turns reused the precise figure instead of re-deriving it.

**Improvement (what memory added):** personalised, context-aware replies with less friction for the user — fewer clarifying questions and more relevant suggestions.

**Limitations observed:**

1. **Token growth / cost** — `ConversationBufferMemory`-style buffers send the *entire* history every turn, so cost and latency grow linearly with conversation length and will eventually hit the context-window limit.
2. **Windowing forgets** — trimming to the last *k* messages fixes cost but silently drops early facts (the Kyoto / vegetarian failure above).
3. **No persistence** — `InMemoryChatMessageHistory` lives in RAM; restarting the kernel wipes everything. A real app would need a database-backed history (Redis, SQL) or a long-term memory store.
4. **Memory is trusted blindly** — if the user (or a model mistake) puts a wrong fact in the history, the agent will keep repeating it.

**Possible next steps:** use summary memory or a pinned key-facts message for long conversations, persist history in a database, and store durable user preferences (diet, home city) in a separate long-term profile rather than in the raw chat log.